<a href="https://colab.research.google.com/github/redinbluesky/handson-llm/blob/main/03_대규모_언어_모델_자세히_살펴보기.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  목차
* [Chapter 3 서론](#chapter3)
* [Chapter 3-1 트랜스포머 모델 개요](#chapter3-1)
    * [Chapter 3-1-1 훈련된 트랜스포머 LLM의 입력과 출력](#chapter3-1-1)
    * [Chapter 3-1-2 정방향 계산의 구성 요소](#chapter3-1-2)

## Chapter 3 서론 <a class="anchor" id="chapter3"></a>
1. 이 장에서 트랜스포머 언어 모델의 작동 방식을 직관적으로 이해해보도록한다.
    - 텍스트 생성 모델이 주요 관심사이므로 특별히 생성형 LLM을 자세히 살펴보도록 한다.

2. 언어 모델을 로드하고 파이프라인 객체를 만들어 텍스트 생성을 위한 준비를 한다.
    - 코드보다는 관려된 개념을 이해하는 데 초점을 맞춘다.

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

#토크나이져를 로드한다.
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")


In [2]:
# 모델을 로드한다.
model = AutoModelForCausalLM.from_pretrained(
	"microsoft/Phi-3-mini-4k-instruct",
	device_map="cuda",
	dtype="auto"
)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [6]:
# 파이프라인 객체를 만든다.
generator = pipeline("text-generation",
                     model=model,
                     tokenizer=tokenizer,
                     return_full_text=False,
                     max_new_tokens=100,
                     do_sample=False
                    )

### Chapter 3-1 트랜스포머 모델 개요 <a class="anchor" id="chapter3-1"></a>
#### Chapter 3-1-1 훈련된 트랜스포머 LLM의 입력과 출력 <a class="anchor" id="chapter3-1-1"></a>
1. 트랜스포머 LLM의 동작 방식을 이해하는 가장 일반적인 방법은 모델을 하나의 소프트웨어 시스템으로 생각하는 것이다.
    - 이 시스템은 텍스트를 입력 받아 응답으로 텍스트를 생성할 수 있다.

2. 아래의 그림은 이메일을 작성할 수 있는 모델의 예를 보여준다.

     ![모델 예시](./image/03_model_example1.png)


3. 이 모델은 한 번에 모든 텍스트를 생성하지 않고 한 번에 하나의 토큰씩 생성한다.
    - 아래의 이미지는 응답으로 네 개의 토큰을 생성하는 과정을 보여준다.
    - 각각의 토큰 생성 과정은 모델을 통과하는 한 번의 정방향계산(forward pass)을 필요로 한다.

        ![토큰 생성 과정](./image/03_token_generation.png)

4. 하나의 토큰을 생성한 후에, 출력 토큰을 입력 프롬프트의 끝에 추가하여 다음 생선 단계를 위한 프롬프트로 사용한다.
    - 출력 토큰이 추가된 프롬프트가 모델에 전달되어 새로운 텍스트를 생성하기 위한 정방향 계산이 수행된다.

        ![프롬프트 업데이트](./image/03_prompt_update.png)

5. 모델은 입력 프롬프트를 기반으로 다음 토큰을 예측하는 작업을 반복하여 텍스트를 생성한다.
    - 이렇게 이전 예측을 사용해 다음 예측을 만드는 모델을 자기회귀 모델(autoregressive model)이라고 한다.

6. 텍스트 생성 LLM은 이 같은 특성이 있어 자기회귀 모델이라고 한다.
    - 이 특성은 자기회귀적이지 않은 BERT와 같은 모델과 구별되는 중요한 특징이다.

In [7]:
# LLM은 한 토큰씩 생성하는 자귀회귀적인 과정을 내부에서 수행한다.
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

output = generator(prompt)

print(output[0]['generated_text'])

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Mention that you've already taken steps to prevent it in the future.


Email to Sarah:

Subject: Sincere Apologies for the Gardening Mishap


Dear Sarah,


I hope this message finds you well. I am writing to express my deepest apologies for the unfortunate incident that occurred in your garden yesterday.


As you know, I have been helping you with your gardening for some time now


#### Chapter 3-1-2 정방향 계산의 구성 요소 <a class="anchor" id="chapter3-1-2"></a>
1. 토크나이저와 언어 모델링 헤드(LM 헤드)는 중용한 구성요소이다.
    - 아래의 그림에서 두 구성요소가 전체 시스템에서 어디에 위치하는지 알 수있다.
    - 토크나이저가 텍스트를 토큰 시퀸스로 변환하고, 트랜스포머 블럭을 통과한 출력 정보를 LM 헤드가 다음 토큰에 대한 확률로 변환한다.

        ![구성 요소](./image/03_components.png)

2. 토크나이저는 토큰의 테이블인 어휘사전을 포함하고 있고, 트랜스포머 모델은 어휘사전에 있는 가 토큰에 연관된 벡터 표현인 토큰 임베딩을 가지고 있다.
    - 50,00개의 토큰으로 구성된 어휘사전과 모델에 있는 임베딩을 그림으로 표현하면 아래와 같다.
    
        ![토크나이저와 임베딩](./image/03_tokenizer_embedding.png)

3. 토큰 하나를 생성하기 위해 토크나이저 -> 트랜스포머 블록 -> LM 헤드를 통과한 후 토큰에 대한 확률을 출력한다.
    - 정방향 계산의 마지막에 모델은 어휘사전에 있는 모든 토큰에 대한 확률 점수를 출력한다.
    
        ![토큰 생성 과정](./image/03_token_generation_process.png)

4. 다양한 종류의 시스템을 만들기 위해 트랜스포머 블록의 스텍에 여러 종류의 헤드를 붙일 수 있다.
    - LM 헤드는 외에 분류 헤드, 회귀 헤드, 시퀀스 레이블링 헤드 등을 붙일 수 있다.

In [ ]:
# model을 출력하여 세부 정보를 확인한다.
#   - 모델 층이 여러 단계로 중첩되어 있으면 가장 상위 층은 model과 lm_head 이다.
#   - 모델의 임베딩 토큰은 32,064개이 이고, 벡터의 크기는 3,072이다.
#   - 가각의 트랜스포머 블록 안에는 어텐션 층과 피드포워드 신경만이 있다.
print(model)

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_featur